In [3]:
!pip install langchain-core langchain langchain-google-genai -U langchain-community chromadb bs4

  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached langchain-1.3.4-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_google_genai-4.2.4-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 8.3 MB/s  0:00:00
   ---------------------------------------- 0.0/832.5 kB ? eta -:--:--
   ---------------------------------------- 832.5/832.5 kB 8.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------------------------- ------------- 1.6/2.4 MB 7.3 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 7.3 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 9.7 MB/s  0:00:00
   ---------------------------------------- 0.0/23.5


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install langchain-text-splitters -U



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


LES IMPORTS

In [2]:
import os
import requests
from bs4 import BeautifulSoup

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


C:\Users\zelha\AppData\Local\Temp\ipykernel_46976\4252901032.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Entre ta clé API Gemini : ")

PARSING VIA BS4 

In [ ]:
URL_BASE = "https://mazerunner.fandom.com"

urls_filtrees = [
    # Personnages principaux
    "/fr/wiki/Thomas",
    "/fr/wiki/Teresa",
    "/fr/wiki/Newt",
    "/fr/wiki/Minho",
    "/fr/wiki/Alby",
    "/fr/wiki/Chuck",
    "/fr/wiki/Gally",
    "/fr/wiki/Brenda",
    "/fr/wiki/Jorge",
    "/fr/wiki/Aris",
    "/fr/wiki/Janson",
    "/fr/wiki/Ava_Paige",
    "/fr/wiki/Vince",
    "/fr/wiki/Mary_Cooper",
    "/fr/wiki/Lawrence",

    # Lieux
    "/fr/wiki/Labyrinthe",
    "/fr/wiki/Terre_Br%C3%BBl%C3%A9e",
    "/fr/wiki/Denver",
    "/fr/wiki/Asheville",
    "/fr/wiki/Bunker",
    "/fr/wiki/Quartier_g%C3%A9n%C3%A9ral_du_WICKED",
    "/fr/wiki/Trou_des_Griffeurs",
    "/fr/wiki/Salle_des_cartes",

    # Créatures et concepts
    "/fr/wiki/Griffeur",
    "/fr/wiki/Fondu",
    "/fr/wiki/Immunis%C3%A9",
    "/fr/wiki/Bo%C3%AEte",
    "/fr/wiki/Effacement",
    "/fr/wiki/Rem%C3%A8de",
    "/fr/wiki/S%C3%A9rum",
    "/fr/wiki/Transformation",
    "/fr/wiki/Braise",

    # Livres et films
    "/fr/wiki/Le_Labyrinthe_(livre)",
    "/fr/wiki/Le_Labyrinthe_(film)",
    "/fr/wiki/La_Terre_Br%C3%BBl%C3%A9e_(livre)",
    "/fr/wiki/La_Terre_Br%C3%BBl%C3%A9e_(film)",
    "/fr/wiki/Le_Rem%C3%A8de_Mortel_(livre)",
    "/fr/wiki/Le_Rem%C3%A8de_Mortel_(film)",
    "/fr/wiki/L%27%C3%89preuve_(s%C3%A9rie)",
]

urls_filtrees = [URL_BASE + url for url in urls_filtrees]
print(f"{len(urls_filtrees)} pages sélectionnées")


39 pages sélectionnées


In [16]:
from urllib.parse import unquote
from langchain_core.documents import Document


URL_API = "https://mazerunner.fandom.com/fr/api.php"

def charger_page_wiki(url):
    page_name = unquote(url.split("/fr/wiki/")[-1])
    
    params = {
        "action": "parse",
        "page": page_name,
        "prop": "text",
        "format": "json"
    }
    
    reponse = requests.get(URL_API, params=params)
    data = reponse.json()
    
    if "error" in data or "parse" not in data:
        return None
    
    html_content = data["parse"]["text"]["*"]
    soup = BeautifulSoup(html_content, "html.parser")
    
    for tag in soup.find_all(["table", "script", "style"]):
        tag.decompose()
    
    paragraphes = soup.find_all("p")
    texte = "\n".join(p.get_text(separator=" ", strip=True) for p in paragraphes if p.get_text(strip=True))
    
    if len(texte) < 500 or "aucun texte sur cette page" in texte.lower():
        print(f"VIDE : {page_name}")
        return None
    
    print(f"OK : {page_name} ({len(texte)} caractères)")
    return Document(page_content=texte, metadata={"source": url, "title": page_name})

documents = []
for url in urls_filtrees:
    doc = charger_page_wiki(url)
    if doc:
        documents.append(doc)
    time.sleep(0.3)

print(f"\n{len(documents)} documents chargés sur {len(urls_filtrees)}")


OK : Thomas (17495 caractères)
OK : Teresa (9052 caractères)
OK : Newt (39255 caractères)
OK : Minho (33427 caractères)
OK : Alby (5758 caractères)
OK : Chuck (4486 caractères)
OK : Gally (7791 caractères)
OK : Brenda (13973 caractères)
OK : Jorge (6137 caractères)
OK : Aris (9807 caractères)
OK : Janson (12392 caractères)
OK : Ava_Paige (4033 caractères)
OK : Vince (1070 caractères)
OK : Mary_Cooper (937 caractères)
OK : Lawrence (2497 caractères)
OK : Labyrinthe (2587 caractères)
OK : Terre_Brûlée (1347 caractères)
OK : Denver (2364 caractères)
OK : Asheville (738 caractères)
OK : Bunker (975 caractères)
OK : Quartier_général_du_WICKED (1642 caractères)
OK : Trou_des_Griffeurs (1361 caractères)
OK : Salle_des_cartes (1087 caractères)
OK : Griffeur (2522 caractères)
OK : Fondu (1008 caractères)
OK : Immunisé (900 caractères)
OK : Boîte (1030 caractères)
OK : Effacement (1140 caractères)
OK : Remède (1761 caractères)
OK : Sérum (644 caractères)
OK : Transformation (945 caractères)
VIDE

In [18]:
documents_propres = []

for doc in documents:
    titre = doc.metadata["title"]
    taille = len(doc.page_content)
    
    if taille < 600:
        print(f"SUPPRIME : {titre} ({taille} chars)")
        print(f"  → {doc.page_content[:100]}\n")
    else:
        documents_propres.append(doc)

print(f"\nDocuments gardés : {len(documents_propres)} / {len(documents)}")



Documents gardés : 38 / 38


CHUNKER POUR PAS AVOIR DES TROP GROS BLOC DE TEXTE

In [19]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " "]
)

chunks = text_splitter.split_documents(documents_propres)

print(f"Nombre de chunks : {len(chunks)}")
print(f"\nExemple de chunk :")
print(chunks[0].page_content)
print(f"\nSource : {chunks[0].metadata['source']}")

Nombre de chunks : 323

Exemple de chunk :
/!\ Attention /!\
Cette page contient des spoilers.
Thomas , auparavant Stephen , est un ancien Blocard du groupe A et un des créateurs du Labyrinthe , tout comme Teresa . Son double du Groupe B est Rachel . Son nom lui a été donné par les créateurs en honneur à Thomas Edison.
Thomas subit une opération (pour lui implanter une puce) pour pouvoir ensuite être envoyé dans le labyrinthe avec les autres blocards . Pendant toute l'opération, Teresa reste à ses côtés, effrayée par ce qui pourrait lui arriver à elle et Thomas. Elle reste cependant calme, étant convaincue que WICKED allait trouver un remède à la braise pour le bien de l'humanité. Thomas est ensuite placé dans la boîte et envoyé dans le labyrinthe.
Il est dit dans l'épilogue que Thomas a été nommé en l'honneur de Thomas Edison après qu'un agent de WICKED ait pointé une ampoule au plafond de la maison de sa mère en disant "Tu sais qui a inventé ce truc, pas vrai ? Peut-être qu'on pourra

EMBEDDING + STOCKAGE DANS CHROMA

In [21]:
BATCH_SIZE = 80
PAUSE = 65

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

total_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"{len(chunks)} chunks → {total_batches} batches de {BATCH_SIZE}\n")

vectorstore = Chroma.from_documents(
    documents=chunks[:BATCH_SIZE],
    embedding=gemini_embeddings,
    persist_directory="./chroma_maze_runner"
)
print(f"Batch 1/{total_batches} OK")

for i in range(BATCH_SIZE, len(chunks), BATCH_SIZE):
    batch_num = i // BATCH_SIZE + 1
    print(f"Pause {PAUSE}s avant batch {batch_num}/{total_batches}...")
    time.sleep(PAUSE)
    vectorstore.add_documents(chunks[i:i+BATCH_SIZE])
    print(f"Batch {batch_num}/{total_batches} OK")

print(f"\nTerminé ! {vectorstore._collection.count()} vecteurs stockés")


323 chunks → 5 batches de 80

Batch 1/5 OK
Pause 65s avant batch 2/5...
Batch 2/5 OK
Pause 65s avant batch 3/5...
Batch 3/5 OK
Pause 65s avant batch 4/5...
Batch 4/5 OK
Pause 65s avant batch 5/5...
Batch 5/5 OK

Terminé ! 323 vecteurs stockés


In [24]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

docs_test = retriever.invoke("Qui est Thomas ?")
print(f"{len(docs_test)} chunks récupérés\n")
for i, doc in enumerate(docs_test):
    print(f"--- Chunk {i+1} ({doc.metadata['title']}) ---")
    print(doc.page_content[:150])
    print()


4 chunks récupérés

--- Chunk 1 (Thomas) ---
/!\ Attention /!\
Cette page contient des spoilers.
Thomas , auparavant Stephen , est un ancien Blocard du groupe A et un des créateurs du Labyrinthe 

--- Chunk 2 (Thomas) ---
Thomas et ses amis élaborent un plan pour s'échapper avec l'aide de Brenda et de Jorge .
Thomas se réveille dans un ascenseur en marche, grimpant vers

--- Chunk 3 (Le_Remède_Mortel_(livre)) ---
Thomas est jeune – il ne sait pas quel âge il a exactement, mais il est jeune. Il est recroquevillé dans un coin, les genoux ramenés contre la poitrin

--- Chunk 4 (Le_Labyrinthe_(livre)) ---
Thomas, 16 ans, se réveille amnésique dans un labyrinthe. Là, il rencontre quelques dizaines d'adolescents qui furent envoyés là à raison d'un par moi

